# 🚖 Análise de Retenção de Motoristas — 99

**Projeto de portfólio | Dados simulados inspirados no contexto real da 99**

---

### Contexto de negócio

A 99 atua no segmento de Ride Hailing, agenciando transporte de passageiros e conectando motoristas parceiros a passageiros via plataforma digital. Uma das maiores preocupações operacionais desse modelo é a **retenção de motoristas**: manter parceiros ativos e engajados é essencial para garantir oferta suficiente e qualidade do serviço.

### Problema

> **O que faz um motorista abandonar a plataforma?**

### Objetivo

Identificar os principais fatores associados ao churn (inatividade) de motoristas, com base em variáveis como avaliação, ganho semanal, frequência de corridas e tempo na plataforma.

---
*⚠️ Os dados utilizados nesta análise são simulados com base no contexto real do mercado de ride hailing no Brasil. Criados exclusivamente para fins de portfólio.*


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Paleta de cores inspirada na 99
AMARELO_99 = "#FFD600"
PRETO_99   = "#1A1A1A"
VERMELHO   = "#E53935"
VERDE      = "#43A047"
AZUL       = "#1E88E5"

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "white",
    "axes.spines.top":  False,
    "axes.spines.right": False,
    "font.family":      "DejaVu Sans",
    "axes.titlesize":   13,
    "axes.labelsize":   11,
})

print("✅ Bibliotecas carregadas com sucesso")


## 1. Geração e Preparação dos Dados

Os dados simulam o perfil de **1.200 motoristas parceiros** distribuídos pelas principais cidades brasileiras onde a 99 opera.

**Variáveis criadas:**
- `cidade` — cidade de operação do motorista
- `tempo_plataforma_meses` — há quanto tempo está cadastrado
- `corridas_por_semana` — frequência média de corridas
- `avaliacao_media` — nota média recebida dos passageiros (1–5)
- `ganho_medio_semanal_r$` — ganho médio semanal em reais
- `cancelamentos_mes` — média de cancelamentos no mês
- `suporte_acionado` — se o motorista já abriu chamado no suporte
- `bonus_recebido` — se recebeu bônus da plataforma no período
- `churn` — **variável alvo**: 1 = motorista inativo, 0 = ativo


In [ ]:
np.random.seed(42)
N = 1200

cidades = ["São Paulo", "Rio de Janeiro", "Belo Horizonte", "Curitiba", "Fortaleza", "Recife"]
pesos   = [0.40, 0.25, 0.15, 0.10, 0.06, 0.04]

df = pd.DataFrame({
    "motorista_id": range(1, N + 1),
    "cidade": np.random.choice(cidades, N, p=pesos),
    "tempo_plataforma_meses": np.random.randint(1, 37, N),
    "corridas_por_semana": np.random.choice(
        [0,1,2,3,4,5,6,7,8,9,10], N,
        p=[0.05,0.08,0.10,0.12,0.14,0.14,0.12,0.10,0.08,0.05,0.02]),
    "avaliacao_media": np.round(np.random.beta(8, 2) * 2 + 3, 1).clip(1, 5),
    "ganho_medio_semanal_r$": np.random.normal(700, 200, N).clip(100, 1800).round(2),
    "cancelamentos_mes": np.random.poisson(2, N),
    "suporte_acionado": np.random.choice([0, 1], N, p=[0.65, 0.35]),
    "bonus_recebido": np.random.choice([0, 1], N, p=[0.45, 0.55]),
})

# Lógica de churn realista baseada em múltiplos fatores
prob_churn = (
    0.05
    + (df["avaliacao_media"] < 4.0).astype(int) * 0.20
    + (df["ganho_medio_semanal_r$"] < 500).astype(int) * 0.25
    + (df["cancelamentos_mes"] > 3).astype(int) * 0.15
    + (df["corridas_por_semana"] <= 2).astype(int) * 0.18
    + (df["suporte_acionado"] == 1).astype(int) * 0.10
    - (df["bonus_recebido"] == 1).astype(int) * 0.12
    - (df["tempo_plataforma_meses"] > 12).astype(int) * 0.08
).clip(0, 0.95)

df["churn"] = (np.random.rand(N) < prob_churn).astype(int)
df["status"] = df["churn"].map({0: "Ativo", 1: "Inativo (Churn)"})

motivos = ["Baixa remuneração", "Migrou p/ concorrente",
           "Problemas com suporte", "Muitos cancelamentos", "Outros"]
df.loc[df["churn"] == 1, "motivo_saida"] = np.random.choice(
    motivos, df["churn"].sum(), p=[0.35, 0.28, 0.17, 0.13, 0.07])
df["motivo_saida"] = df["motivo_saida"].fillna("—")

print(f"Dataset criado com {df.shape[0]} linhas e {df.shape[1]} colunas")
df.head()


## 2. Análise Exploratória Inicial

In [ ]:
# Visão geral do dataset
print("=== TIPOS E NULOS ===")
print(df.dtypes)
print("\nValores nulos:")
print(df.isnull().sum())


In [ ]:
# Estatísticas descritivas
df.describe().round(2)


In [ ]:
# Distribuição do churn
total = len(df)
churn_count = df["churn"].sum()
ativo_count = total - churn_count

print("=== DISTRIBUIÇÃO DE STATUS ===")
print(df["status"].value_counts())
print(f"\nTaxa de churn: {churn_count/total*100:.1f}%")
print(f"Taxa de retenção: {ativo_count/total*100:.1f}%")


## 3. Visualizações — Visão Geral

O painel abaixo apresenta quatro dimensões da análise de retenção:
1. **Distribuição geral** de ativos vs. churn
2. **Avaliação média** como fator de retenção
3. **Taxa de churn por cidade**
4. **Relação entre corridas, ganho e churn**


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Análise de Retenção de Motoristas — Dados Simulados 99",
             fontsize=16, fontweight="bold", color=PRETO_99, y=1.01)

# 1. Pizza — distribuição de status
ax = axes[0, 0]
contagem = df["status"].value_counts()
wedges, texts, autotexts = ax.pie(
    contagem, labels=contagem.index, autopct="%1.1f%%",
    colors=[VERDE, VERMELHO], startangle=90,
    wedgeprops=dict(edgecolor="white", linewidth=2))
for at in autotexts:
    at.set_fontsize(12); at.set_fontweight("bold"); at.set_color("white")
ax.set_title("Distribuição de Status dos Motoristas", pad=12)

# 2. Boxplot — avaliação vs churn
ax = axes[0, 1]
grupos = [df[df["churn"]==0]["avaliacao_media"], df[df["churn"]==1]["avaliacao_media"]]
bp = ax.boxplot(grupos, patch_artist=True, widths=0.5,
                medianprops=dict(color=PRETO_99, linewidth=2))
bp["boxes"][0].set_facecolor(VERDE); bp["boxes"][1].set_facecolor(VERMELHO)
ax.set_xticks([1, 2]); ax.set_xticklabels(["Ativo", "Inativo (Churn)"])
ax.set_ylabel("Avaliação Média"); ax.set_title("Avaliação Média × Status")
ax.axhline(4.0, color=AMARELO_99, linewidth=1.5, linestyle="--", label="Nota 4.0")
ax.legend(fontsize=9)

# 3. Barras horizontais — churn por cidade
ax = axes[1, 0]
churn_cidade = df.groupby("cidade")["churn"].mean().sort_values(ascending=True) * 100
bars = ax.barh(churn_cidade.index, churn_cidade.values,
               color=[VERMELHO if v > churn_cidade.mean() else AZUL for v in churn_cidade.values])
ax.axvline(churn_cidade.mean(), color=PRETO_99, linewidth=1.5,
           linestyle="--", label=f"Média: {churn_cidade.mean():.1f}%")
ax.set_xlabel("Taxa de Churn (%)"); ax.set_title("Taxa de Churn por Cidade")
ax.legend(fontsize=9)
for bar, val in zip(bars, churn_cidade.values):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
            f"{val:.1f}%", va="center", fontsize=9)

# 4. Scatter — corridas vs ganho
ax = axes[1, 1]
cores_scatter = [VERDE if c == 0 else VERMELHO for c in df["churn"]]
ax.scatter(df["corridas_por_semana"], df["ganho_medio_semanal_r$"],
           c=cores_scatter, alpha=0.35, s=20)
ax.set_xlabel("Corridas por Semana"); ax.set_ylabel("Ganho Médio Semanal (R$)")
ax.set_title("Corridas × Ganho por Status")
patch1 = mpatches.Patch(color=VERDE, label="Ativo")
patch2 = mpatches.Patch(color=VERMELHO, label="Churn")
ax.legend(handles=[patch1, patch2], fontsize=9)

plt.tight_layout()
plt.show()


## 4. Fatores que Influenciam o Churn

Analisamos três dimensões específicas:
- **Motivos declarados** de saída da plataforma
- **Faixa de ganho semanal** e sua relação com churn
- **Tempo na plataforma** — quando o motorista tem mais chance de sair?


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Fatores que Influenciam o Churn de Motoristas",
             fontsize=15, fontweight="bold", color=PRETO_99)

# 1. Motivos de saída
ax = axes[0]
motivos_count = df[df["churn"]==1]["motivo_saida"].value_counts()
cores_m = [AMARELO_99, VERMELHO, AZUL, "#FF8F00", "#8E24AA"]
bars = ax.bar(range(len(motivos_count)), motivos_count.values, color=cores_m)
ax.set_xticks(range(len(motivos_count)))
ax.set_xticklabels(motivos_count.index, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Nº de Motoristas"); ax.set_title("Motivos de Saída da Plataforma")
for bar, val in zip(bars, motivos_count.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(val), ha="center", fontsize=9, fontweight="bold")

# 2. Churn por faixa de ganho
ax = axes[1]
bins = [0, 400, 600, 800, 1000, 1800]
labels_ganho = ["<400", "400–600", "600–800", "800–1k", ">1k"]
df["faixa_ganho"] = pd.cut(df["ganho_medio_semanal_r$"], bins=bins, labels=labels_ganho)
churn_ganho = df.groupby("faixa_ganho", observed=True)["churn"].mean() * 100
bars = ax.bar(churn_ganho.index, churn_ganho.values,
              color=[VERMELHO if v > 40 else AZUL for v in churn_ganho.values])
ax.set_xlabel("Ganho Médio Semanal (R$)"); ax.set_ylabel("Taxa de Churn (%)")
ax.set_title("Churn por Faixa de Ganho")
for bar, val in zip(bars, churn_ganho.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{val:.1f}%", ha="center", fontsize=9, fontweight="bold")

# 3. Churn por tempo na plataforma
ax = axes[2]
bins_t = [0, 3, 6, 12, 24, 36]
labels_t = ["0–3m", "3–6m", "6–12m", "12–24m", "24–36m"]
df["faixa_tempo"] = pd.cut(df["tempo_plataforma_meses"], bins=bins_t, labels=labels_t)
churn_tempo = df.groupby("faixa_tempo", observed=True)["churn"].mean() * 100
ax.plot(churn_tempo.index, churn_tempo.values,
        marker="o", color=AMARELO_99, linewidth=2.5, markersize=8,
        markerfacecolor=PRETO_99)
ax.fill_between(range(len(churn_tempo)), churn_tempo.values, alpha=0.15, color=AMARELO_99)
ax.set_xticks(range(len(churn_tempo))); ax.set_xticklabels(churn_tempo.index)
ax.set_ylabel("Taxa de Churn (%)"); ax.set_title("Churn por Tempo na Plataforma")
for i, val in enumerate(churn_tempo.values):
    ax.text(i, val + 1, f"{val:.1f}%", ha="center", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.show()


## 5. Correlação entre Variáveis e Churn

O heatmap mostra como cada variável se relaciona com as demais. Valores próximos de **+1** indicam correlação positiva e próximos de **-1** indicam correlação negativa com o churn.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
cols_num = ["churn", "avaliacao_media", "ganho_medio_semanal_r$",
            "corridas_por_semana", "cancelamentos_mes",
            "tempo_plataforma_meses", "suporte_acionado", "bonus_recebido"]
corr = df[cols_num].corr()
labels_pt = ["Churn", "Avaliação", "Ganho Sem.", "Corridas/Sem.",
             "Cancelamentos", "Tempo Plataf.", "Suporte", "Bônus"]
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdYlGn",
            center=0, ax=ax, xticklabels=labels_pt, yticklabels=labels_pt,
            linewidths=0.5, cbar_kws={"shrink": 0.8})
ax.set_title("Correlação entre Variáveis e Churn", fontsize=14, fontweight="bold", pad=15)
plt.tight_layout()
plt.show()


## 6. Conclusões e Recomendações

### 📌 Principais achados

| Fator | Impacto no Churn |
|---|---|
| Ganho semanal abaixo de R$ 500 | Alto — principal driver de saída |
| Avaliação média abaixo de 4.0 | Moderado — motoristas mal avaliados saem mais |
| Menos de 3 corridas por semana | Alto — baixo engajamento prediz inatividade |
| Primeiros 6 meses na plataforma | Período crítico — maior taxa de abandono |
| Recebimento de bônus | Protetor — reduz significativamente o churn |
| Suporte acionado | Moderado — indica insatisfação com a plataforma |

### 💡 Recomendações de negócio

1. **Programa de onboarding reforçado** nos primeiros 6 meses — período de maior risco de churn
2. **Alertas proativos** para motoristas com ganho semanal caindo abaixo de R$ 500
3. **Política de bônus** como alavanca de retenção — dados mostram que bônus reduz churn
4. **Melhoria no fluxo de suporte** — motoristas que acionam suporte têm maior propensão ao churn, sugerindo que o atendimento pode não estar resolvendo o problema

### 🔎 Próximos passos sugeridos
- Aplicar modelo de classificação (Logistic Regression ou Random Forest) para prever churn individualmente
- Criar score de risco por motorista para ações preventivas
- Analisar sazonalidade: há períodos do ano com maior churn?

---
*Projeto desenvolvido por Yasmin Guedes — Portfólio de Transição para Dados*  
*Dados simulados para fins educacionais e de portfólio*
